In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# 1. MENU PRINCIPAL
def mostrar_menu():
    print("\n" + "=" * 55)
    print("       SISTEMA DE PREDICCION DE VALORES DE VIVIENDA")
    print("=" * 55)
    print("1. Evaluar una zona de California")
    print("2. Evaluar una zona de Boston")
    print("3. Salir")
    print("=" * 55)



# 2. FUNCIONES PARA VALIDAR LAS RESPUESTAS
def pedir_numero(pregunta, minimo=None, maximo=None, entero=False):
    while True:
        try:
            respuesta = input(pregunta).strip().replace(",", ".")
            valor = float(respuesta)

            if not np.isfinite(valor):
                print("Error: introduce un numero valido.")
                continue

            if entero and not valor.is_integer():
                print("Error: debes introducir un numero entero.")
                continue

            if minimo is not None and valor < minimo:
                print(f"El valor debe ser como minimo {minimo}.")
                continue

            if maximo is not None and valor > maximo:
                print(f"El valor no puede ser mayor que {maximo}.")
                continue

            return int(valor) if entero else valor

        except ValueError:
            print("Respuesta incorrecta. Introduce solamente un numero.")


def pedir_si_no(pregunta):
    while True:
        respuesta = input(pregunta + " (1 = Si / 0 = No): ").strip().lower()

        if respuesta in ["1", "si", "sí", "s"]:
            return 1

        if respuesta in ["0", "no", "n"]:
            return 0

        print("Respuesta incorrecta. Escribe 1 para Si o 0 para No.")


# 3. CARGAR DATASET CALIFORNIA HOUSING
def cargar_california():

    print("\nCargando dataset California Housing...")

    datos = fetch_california_housing(as_frame=True)

    X = datos.data
    y = datos.target

    # El objetivo esta expresado en unidades de 100,000 USD.
    factor = 100000

    return X, y, factor


# 4. CARGAR DATASET BOSTON HOUSING
def cargar_boston():

    print("\nCargando dataset historico Boston Housing...")

    url = "https://lib.stat.cmu.edu/datasets/boston"

    columnas = [
        "CRIM", "ZN", "INDUS", "CHAS", "NOX",
        "RM", "AGE", "DIS", "RAD", "TAX",
        "PTRATIO", "B", "LSTAT"
    ]

    datos_originales = pd.read_csv(
        url,
        sep=r"\s+",
        skiprows=22,
        header=None
    )

    # Reconstruir las 506 filas del dataset.
    datos = np.hstack([
        datos_originales.values[::2, :],
        datos_originales.values[1::2, :2]
    ])

    objetivo = datos_originales.values[1::2, 2]

    df = pd.DataFrame(datos, columns=columnas)

    # MEDV representa el valor mediano de las viviendas.
    df["MEDV"] = objetivo

    # Se excluye B por los problemas eticos asociados
    # a esta variable historica.
    X = df.drop(columns=["B", "MEDV"])
    y = df["MEDV"]

    # MEDV esta expresado en miles de dolares.
    factor = 1000

    return X, y, factor


# 5. SELECCIONAR ALGORITMO DE MACHINE LEARNING
def seleccionar_algoritmo():

    print("\nSelecciona el algoritmo que deseas utilizar:")
    print("1. Regresion Lineal")
    print("2. Random Forest")

    while True:
        opcion = input("Selecciona una opcion: ").strip()

        if opcion == "1":
            modelo = Pipeline([
                ("escalador", StandardScaler()),
                ("regresion", LinearRegression())
            ])

            print("\nAlgoritmo seleccionado: Regresion Lineal")
            return modelo

        elif opcion == "2":
            modelo = RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )

            print("\nAlgoritmo seleccionado: Random Forest")
            return modelo

        else:
            print("Opcion incorrecta. Selecciona 1 o 2.")


# 6. PREGUNTAS PARA CALIFORNIA HOUSING
def solicitar_california():

    print("\n" + "=" * 55)
    print("       DATOS DE LA ZONA - CALIFORNIA")
    print("=" * 55)

    print("\nResponde las siguientes preguntas.")
    print("Introduce los valores numericos solicitados.\n")

    respuestas = {}

    # MedInc: ingreso medio en decenas de miles de USD.
    ingreso = pedir_numero(
        "1. ¿Cual es el ingreso anual medio de los hogares? (USD): $",
        minimo=0
    )

    respuestas["MedInc"] = ingreso / 10000

    # HouseAge
    respuestas["HouseAge"] = pedir_numero(
        "\n2. ¿Cual es la antiguedad media de las viviendas? (años): ",
        minimo=0,
        maximo=100
    )

    # AveRooms
    respuestas["AveRooms"] = pedir_numero(
        "\n3. ¿Cuantas habitaciones tiene en promedio cada vivienda?: ",
        minimo=0.01
    )

    # AveBedrms
    respuestas["AveBedrms"] = pedir_numero(
        "\n4. ¿Cuantos dormitorios tiene en promedio cada vivienda?: ",
        minimo=0.01
    )

    # Population
    respuestas["Population"] = pedir_numero(
        "\n5. ¿Cuantas personas viven aproximadamente en la zona?: ",
        minimo=0,
        entero=True
    )

    # AveOccup
    respuestas["AveOccup"] = pedir_numero(
        "\n6. ¿Cuantas personas ocupan en promedio cada vivienda?: ",
        minimo=0.01
    )

    # Latitude
    respuestas["Latitude"] = pedir_numero(
        "\n7. ¿Cual es la latitud de la ubicacion? (entre -90 y 90): ",
        minimo=-90,
        maximo=90
    )

    # Longitude
    respuestas["Longitude"] = pedir_numero(
        "\n8. ¿Cual es la longitud de la ubicacion? (entre -180 y 180): ",
        minimo=-180,
        maximo=180
    )

    return respuestas


# 7. PREGUNTAS PARA BOSTON HOUSING

def solicitar_boston():

    print("\n" + "=" * 55)
    print("        DATOS DE LA ZONA - BOSTON")
    print("=" * 55)

    print("\nResponde las siguientes preguntas.")
    print("Algunas caracteristicas utilizan indices historicos.\n")

    respuestas = {}

    # CRIM
    respuestas["CRIM"] = pedir_numero(
        "1. ¿Cual es el indice de criminalidad registrado en la zona?: ",
        minimo=0
    )

    # ZN
    respuestas["ZN"] = pedir_numero(
        "\n2. ¿Que porcentaje de terreno esta reservado para grandes "
        "lotes residenciales? (%): ",
        minimo=0,
        maximo=100
    )

    # INDUS
    respuestas["INDUS"] = pedir_numero(
        "\n3. ¿Que porcentaje de terreno corresponde a actividades "
        "industriales? (%): ",
        minimo=0,
        maximo=100
    )

    # CHAS
    respuestas["CHAS"] = pedir_si_no(
        "\n4. ¿La zona limita con el rio Charles?"
    )

    # NOX
    respuestas["NOX"] = pedir_numero(
        "\n5. ¿Cual es el indice de concentracion de oxidos nitr icos "
        "(NOX) del dataset? (entre 0 y 1): ".replace("nitr icos", "nitr icos"),
        minimo=0,
        maximo=1
    )

    # RM
    respuestas["RM"] = pedir_numero(
        "\n6. ¿Cuantas habitaciones tiene en promedio cada vivienda?: ",
        minimo=0.01
    )

    # AGE
    respuestas["AGE"] = pedir_numero(
        "\n7. ¿Que porcentaje de viviendas fue construido antes de 1940? (%): ",
        minimo=0,
        maximo=100
    )

    # DIS
    respuestas["DIS"] = pedir_numero(
        "\n8. ¿Cual es la distancia ponderada a los centros de empleo? "
        "(indice del dataset): ",
        minimo=0.01
    )

    # RAD
    respuestas["RAD"] = pedir_numero(
        "\n9. ¿Cual es el indice de acceso a carreteras principales? "
        "(numero entero): ",
        minimo=1,
        maximo=24,
        entero=True
    )

    # TAX
    respuestas["TAX"] = pedir_numero(
        "\n10. ¿Cual es el indice historico del impuesto inmobiliario "
        "por cada 10,000 USD?: ",
        minimo=0
    )

    # PTRATIO
    respuestas["PTRATIO"] = pedir_numero(
        "\n11. ¿Cuantos alumnos hay por cada profesor aproximadamente?: ",
        minimo=0.01
    )

    # LSTAT
    respuestas["LSTAT"] = pedir_numero(
        "\n12. Introduce el porcentaje del indicador socioeconomico "
        "LSTAT del dataset (%): ",
        minimo=0,
        maximo=100
    )

    return respuestas


# 8. ENTRENAMIENTO Y EVALUACION DEL MODELO

def entrenar_modelo(X, y, modelo, factor):

    print("\nEntrenando el modelo de Machine Learning...")
    print("Dividiendo los datos en entrenamiento y prueba...")

    X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    modelo.fit(X_entrenamiento, y_entrenamiento)

    predicciones = modelo.predict(X_prueba)

    # Calcular metricas.
    mae = mean_absolute_error(y_prueba, predicciones)
    mse = mean_squared_error(y_prueba, predicciones)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_prueba, predicciones)

    print("\n" + "=" * 55)
    print("             RESULTADOS DE EVALUACION")
    print("=" * 55)

    print(f"MAE  : ${mae * factor:,.2f} USD")
    print(f"MSE  : ${mse * (factor ** 2):,.2f} USD²")
    print(f"RMSE : ${rmse * factor:,.2f} USD")
    print(f"R²   : {r2:.4f}")

    print("=" * 55)

    return modelo


# 9. REALIZAR PREDICCION
def realizar_prediccion(modelo, respuestas, columnas, factor):

    # Organizar las respuestas en el orden que espera el dataset.
    entrada = pd.DataFrame(
        [respuestas],
        columns=columnas
    )

    prediccion = modelo.predict(entrada)[0]

    valor_estimado = prediccion * factor

    print("\n" + "=" * 55)
    print("             RESULTADO DE LA PREDICCION")
    print("=" * 55)

    if valor_estimado < 0:
        print("El modelo produjo una estimacion negativa.")
        print("Este resultado no es interpretable como un valor de vivienda.")
    else:
        print(f"Valor mediano estimado: ${valor_estimado:,.2f} USD")

    print("\nEste resultado es una estimacion basada en los datos")
    print("historicos de la zona, no una tasacion de una casa individual.")

    print("=" * 55)


# 10. PROGRAMA PRINCIPAL

def main():

    while True:

        mostrar_menu()

        opcion = input("Selecciona una opcion: ").strip()

        try:

            if opcion == "1":

                X, y, factor = cargar_california()

                modelo = seleccionar_algoritmo()

                modelo = entrenar_modelo(
                    X, y, modelo, factor
                )

                respuestas = solicitar_california()

                realizar_prediccion(
                    modelo,
                    respuestas,
                    X.columns,
                    factor
                )

            elif opcion == "2":

                X, y, factor = cargar_boston()

                modelo = seleccionar_algoritmo()

                modelo = entrenar_modelo(
                    X, y, modelo, factor
                )

                respuestas = solicitar_boston()

                realizar_prediccion(
                    modelo,
                    respuestas,
                    X.columns,
                    factor
                )

            elif opcion == "3":

                print("\nGracias por utilizar el programa.")
                break

            else:

                print("\nOpcion incorrecta. Selecciona 1, 2 o 3.")

        except Exception as error:

            print("\nNo fue posible completar la evaluacion.")
            print("Detalle del error:", error)


# 11. EJECUTAR PROGRAMA

if __name__ == "__main__":
    main()



       SISTEMA DE PREDICCION DE VALORES DE VIVIENDA
1. Evaluar una zona de California
2. Evaluar una zona de Boston
3. Salir
Selecciona una opcion: 1

Cargando dataset California Housing...

Selecciona el algoritmo que deseas utilizar:
1. Regresion Lineal
2. Random Forest
Selecciona una opcion: 2

Algoritmo seleccionado: Random Forest

Entrenando el modelo de Machine Learning...
Dividiendo los datos en entrenamiento y prueba...

             RESULTADOS DE EVALUACION
MAE  : $32,681.19 USD
MSE  : $2,539,759,249.19 USD²
RMSE : $50,396.02 USD
R²   : 0.8062

       DATOS DE LA ZONA - CALIFORNIA

Responde las siguientes preguntas.
Introduce los valores numericos solicitados.

1. ¿Cual es el ingreso anual medio de los hogares? (USD): $2000

2. ¿Cual es la antiguedad media de las viviendas? (años): 20

3. ¿Cuantas habitaciones tiene en promedio cada vivienda?: 6

4. ¿Cuantos dormitorios tiene en promedio cada vivienda?: 2

5. ¿Cuantas personas viven aproximadamente en la zona?: 5

6. ¿Cuantas